In [12]:
!pip uninstall -y pillow

Found existing installation: pillow 11.3.0
Uninstalling pillow-11.3.0:
  Successfully uninstalled pillow-11.3.0


In [13]:
!pip install --no-cache-dir \
    pillow==11.3.0 \
    transformers==4.52.4 \
    sentence-transformers==4.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 66.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pdfplumber 0.11.10 requires Pillow>=12.2.0, but you have pillow 11.3.0 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [1]:
!pip install pymupdf pytesseract

In [2]:
import fitz
import pytesseract
from PIL import Image
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def extract_text(pdf_path):

    doc = fitz.open(pdf_path)
    complete_text = ""

    for page_num, page in enumerate(doc):
        # Render page at high resolution
        pix = page.get_pixmap(matrix=fitz.Matrix(4,4))

        img = Image.frombytes(
            "RGB",
            [pix.width, pix.height],
            pix.samples
        )

        # ---------- First OCR Attempt ----------
        page_text = pytesseract.image_to_string(
            img,
            config="--oem 3 --psm 3"
        )

        # ---------- If OCR quality is poor ----------
        if len(page_text.strip()) < 100:

            page_text = pytesseract.image_to_string(
                img,
                config="--oem 3 --psm 6"
            )

        complete_text += page_text + "\n"

    doc.close()

    return complete_text


# ==========================================================
# TEXT CLEANING
# ==========================================================

def clean_text(text):

    text = text.replace("“", '"')
    text = text.replace("”", '"')
    text = text.replace("‘", "'")
    text = text.replace("’", "'")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)

    text = re.sub(
        r"Page\s+\d+\s+of\s+\d+",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"SET-\d+",
        "",
        text,
        flags=re.IGNORECASE
    )
    return text


# ==========================================================
# SUBJECT DETECTION
# ==========================================================

def detect_subject(text):
    sample = text[:5000].lower()

    english_keywords = [
        "read the passage",
        "writing skills",
        "grammar",
        "letter",
        "notice",
        "poem",
        "literature",
        "comprehension",
        "article"
    ]

    math_keywords = [
        "quadratic",
        "triangle",
        "find the value",
        "solve",
        "probability",
        "polynomial",
        "trigonometry",
        "surface area",
        "coordinate geometry",
        "circle"
    ]

    english_score = 0
    math_score = 0
    for word in english_keywords:
        if word in sample:
            english_score += 1
    for word in math_keywords:
        if word in sample:
            math_score += 1
    if math_score > english_score:
        return "Math"
    return "English"


# ==========================================================
# QUESTION EXTRACTION
# ==========================================================

def extract_questions(text):

    block_pattern = r"(?=^\s*\d+\.\s*)"
    blocks = re.split(
        block_pattern,
        text,
        flags=re.MULTILINE
    )

    parsed_questions = []
    q_pattern = r"^\s*(\d+\.)"
    marks_pattern = r"""
        (
            \d+\s*[Mm]arks?
            |
            \d+\s*[xX×]\s*\d+\s*=\s*\d+
            |
            \d+\s*\(\d+\)
        )
    """

    for block in blocks:
        block = block.strip()

        if len(block) < 5:
            continue

        match = re.match(
            q_pattern,
            block
        )

        if match is None:
            continue
        q_num = match.group(1)

        mark = re.search(
            marks_pattern,
            block,
            flags=re.IGNORECASE | re.VERBOSE
        )

        if mark:
            marks = mark.group(0)
        else:
            marks = "N/A"

        question = re.sub(
            q_pattern,
            "",
            block
        )

        question = re.sub(
            marks_pattern,
            "",
            question,
            flags=re.IGNORECASE | re.VERBOSE
        )

        question = re.sub(
            r"\s+",
            " ",
            question
        ).strip()

        parsed_questions.append({
            "question_number": q_num,
            "question": question,
            "marks": marks
        })
    return parsed_questions


# ==========================================================
# ENGLISH TOPICS
# ==========================================================

english_topics = [
    "Reading Comprehension",
    "Case Based Passage",
    "Writing Skills",
    "Notice Writing",
    "Letter Writing",
    "Article Writing",
    "Report Writing",
    "Grammar",
    "Gap Filling",
    "Editing",
    "Literature",
    "Poetry",
    "Prose",
    "Character Sketch",
    "Long Answer",
    "Short Answer"
]

# ==========================================================
# MATH TOPICS
# ==========================================================

math_topics = [
    "Real Numbers",
    "Polynomials",
    "Pair of Linear Equations",
    "Quadratic Equations",
    "Arithmetic Progression",
    "Triangles",
    "Coordinate Geometry",
    "Circles",
    "Trigonometry",
    "Applications of Trigonometry",
    "Surface Area and Volume",
    "Statistics",
    "Probability",
    "Case Study",
    "Assertion Reason"
]

In [3]:
print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model Loaded Successfully.\n")

def classify_questions(parsed_questions, topics):
    if len(parsed_questions) == 0:
        print("No questions found.")
        return

    question_texts = [
        q["question"]
        for q in parsed_questions
    ]


    question_numbers = [
        q["question_number"]
        for q in parsed_questions
    ]

    marks = [
        q["marks"]
        for q in parsed_questions
    ]
    print("Generating Question Embeddings...")

    question_embeddings = model.encode(
        question_texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    topic_embeddings = model.encode(
        topics,
        normalize_embeddings=True
    )

    return (
        question_texts,
        question_numbers,
        marks,
        question_embeddings,
        topic_embeddings
    )


# pdf_path = "/content/30-8-3_Maths Std_E2_10_2026.pdf"

# For English
pdf_path = "/content/English(Comm)_E2_X_2026.pdf"
print("Extracting Text from PDF...\n")
text = extract_text(pdf_path)
print("Cleaning OCR Text...\n")
text = clean_text(text)
print("Detecting Subject...\n")
subject = detect_subject(text)
print(f"Detected Subject : {subject}")
print()
print("Extracting Questions...\n")
questions = extract_questions(text)
print(f"Questions Extracted : {len(questions)}")
print()

# Show first few extracted questions
for q in questions[:5]:
    print("-" * 60)
    print(q["question_number"])
    print()
    print(q["question"][:250])
    print()
print()

# Select Topic List Automatically

if subject == "English":
    topics = english_topics

elif subject == "Math":
    topics = math_topics
else:
    topics = []

# Run Similarity

(
    question_texts,
    question_numbers,
    marks,
    question_embeddings,
    topic_embeddings
) = classify_questions(
    questions,
    topics
)

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Model Loaded Successfully.

Extracting Text from PDF...

Cleaning OCR Text...

Detecting Subject...

Detected Subject : English

Extracting Questions...

Questions Extracted : 12

------------------------------------------------------------
1.

Read the following passage carefully : 12M (I) The Nile River, the longest river in the world, flows from South to North through Northeastern Africa. It begins from the rivers that flow into Lake Victoria (located in modern day Kenya, Tanzania and Ug

------------------------------------------------------------
2.

Read the following passage carefully : 10 M (I) In today's digital age, teenagers are spending increasing amount of time on screens, whether for school, social media, gaming, or entertainment. According to a 2024 nationwide survey by the Indian Digit

------------------------------------------------------------
6.

(A) Write an article in about 150 words, as Kumud / Kavish, Class-X, stressing the importance of "environmental conservat

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [4]:

print("=" * 80)
print("STEP 7 : GENERATING TOPIC EMBEDDINGS")
print("=" * 80)

# Generate embeddings only once for all predefined topics
topic_embeddings = model.encode(
    topics,
    normalize_embeddings=True,
    show_progress_bar=False
)

print(f"Total Topics : {len(topics)}")
print(f"Topic Embedding Shape : {topic_embeddings.shape}")

print("\nTopic embeddings generated successfully.\n")

STEP 7 : GENERATING TOPIC EMBEDDINGS
Total Topics : 16
Topic Embedding Shape : (16, 384)

Topic embeddings generated successfully.



In [5]:
# ==========================================================
# STEP 8 : COMPUTE COSINE SIMILARITY
# ==========================================================

print("=" * 80)
print("STEP 8 : COMPUTING COSINE SIMILARITY")
print("=" * 80)

question_embeddings = model.encode(
    question_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

all_similarity_scores = []

for idx, q_embedding in enumerate(question_embeddings):
    similarity_scores = cosine_similarity(
        q_embedding.reshape(1, -1),
        topic_embeddings
    )[0]

    all_similarity_scores.append(similarity_scores)
print("Cosine similarity computed successfully.\n")

STEP 8 : COMPUTING COSINE SIMILARITY


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Cosine similarity computed successfully.



In [6]:
# ==========================================================
# STEP 9 : TOPIC CLASSIFICATION
# ==========================================================

print("=" * 80)
print("STEP 9 : CLASSIFY QUESTIONS")
print("=" * 80)

classification_results = []

for idx, scores in enumerate(all_similarity_scores):
    topic_scores = []

    for topic_idx, topic in enumerate(topics):
        topic_scores.append({
            "topic": topic,
            "confidence": round(
                float(scores[topic_idx]),
                3
            )
        })

    topic_scores = sorted(
        topic_scores,
        key=lambda x: x["confidence"],
        reverse=True
    )
    prediction = {
        "question_number": question_numbers[idx],
        "question": question_texts[idx],
        "predicted_topic": topic_scores[0]["topic"],
        "confidence": topic_scores[0]["confidence"],
        "top_3_predictions": topic_scores[:3]
    }

    classification_results.append(prediction)

print(f"Successfully classified {len(classification_results)} questions.\n")

STEP 9 : CLASSIFY QUESTIONS
Successfully classified 12 questions.



In [7]:
for result in classification_results:

    print("=" * 80)
    print("Question :", result["question_number"])
    print(result["question"])
    print()

    print("Predicted Topic :", result["predicted_topic"])
    print("Confidence :", result["confidence"])
    print()

    print("Top 3 Predictions")
    for item in result["top_3_predictions"]:
        print(f"  {item['topic']} --> {item['confidence']}")

Question : 1.
Read the following passage carefully : 12M (I) The Nile River, the longest river in the world, flows from South to North through Northeastern Africa. It begins from the rivers that flow into Lake Victoria (located in modern day Kenya, Tanzania and Uganda) and travels more than 6,800 kilometers (4,000 miles) to the North, emptying into the Mediterranean Sea on Egypt's coast. (II) The river's three main tributaries are the Atbara, the Blue Nile and the White Nile. The entire Nile River basin — made up of interconnected streams, lakes and rivers — threads its way through 11 African countries : Burundi, Democratic Republic of the Congo, Egypt, Eritrea, Ethiopia, Kenya, Rwanda, South Sudan, Sudan, Tanzania and Uganda. 1 * Page 2of 12 * 11 WNL (III) The Nile River was critical to the development of ancient Egypt. The soil of the Nile River Delta between Cairo, Egypt and the Mediterranean Sea is rich in nutrients, due to the large silt deposits the Nile leaves behind as it flows

In [8]:
import pandas as pd

df = pd.DataFrame(classification_results)
df.to_csv("classification_results.csv", index=False)